### step 1 Data Integration

In [20]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [46]:
def process_all_pdf(pdf_dirs):
    all_docs=[]
    pdf_dir=Path(pdf_dirs)
    pdf_files=list(pdf_dir.glob("*.pdf"))
    print(len(pdf_files))
    for file in pdf_files:
        try:
            loader=PyMuPDFLoader(str(file))
            document=loader.load()
            for doc in document:
                doc.metadata['source_file']=file.name
                doc.metadata['file_type']='pdf'
                
            all_docs.extend(document)
        except Exception as e:
            print(e)
            
    return all_docs

all_pdf_docs=process_all_pdf("data/PDF")
all_pdf_docs

4


[Document(metadata={'producer': 'WeasyPrint 68.1', 'creator': '', 'creationdate': '', 'source': 'data\\PDF\\first.pdf', 'file_path': 'data\\PDF\\first.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'first.pdf', 'file_type': 'pdf'}, page_content='Price Per Token Guide to Using Fewer\nTokens\nMost developers use 2-5x more tokens than they need to. Here\'s how to fix that.\nThis guide covers eight practical ways to cut your token usage. Each one is independent\n— you can start with whichever applies to you — but they compound when used together.\nTip 1: Trim Your System Prompt\nYour system prompt gets sent with every single API call. There\'s no getting around that if\nyou\'re using an LLM as an assistant because each call is stateless- all the context needs\nto be included in every input. It is worth spending time on a frequently used system p

### step 2 Chunking

In [47]:
def split_docs(documents,chunk_size=1000,chunk_overlap=200):
    text_sp=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n','\n',' ','']
    )
    split_docs=text_sp.split_documents(documents)
    return split_docs

chunks=split_docs(documents=all_pdf_docs)
chunks

[Document(metadata={'producer': 'WeasyPrint 68.1', 'creator': '', 'creationdate': '', 'source': 'data\\PDF\\first.pdf', 'file_path': 'data\\PDF\\first.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'first.pdf', 'file_type': 'pdf'}, page_content='Price Per Token Guide to Using Fewer\nTokens\nMost developers use 2-5x more tokens than they need to. Here\'s how to fix that.\nThis guide covers eight practical ways to cut your token usage. Each one is independent\n— you can start with whichever applies to you — but they compound when used together.\nTip 1: Trim Your System Prompt\nYour system prompt gets sent with every single API call. There\'s no getting around that if\nyou\'re using an LLM as an assistant because each call is stateless- all the context needs\nto be included in every input. It is worth spending time on a frequently used system p

### step 3 Embedding


In [23]:
import uuid
import numpy as np
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import chromadb
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [25]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
        except Exception as e:
            print(e)
        
    
    def generate_embedding(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("model not loaded")
        emb=self.model.encode(texts,show_progress_bar=True)
        return emb
    
emb_manager = EmbeddingManager()
emb_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3466.01it/s]


In [48]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",per_dir:str='data/vector_store'):
        self.collection_name=collection_name
        self.per_dir=per_dir
        self.client=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.per_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.per_dir)
            
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'description':'bla bla bla'}
            )
        except Exception:
            print(Exception)
            raise
    
    def add_elements(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("values mis match")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
        except Exception as e:
            print(e)
            raise
        
        

In [49]:
vectorstore=VectorStore()
texts=[doc.page_content for doc in chunks]
embeddings=emb_manager.generate_embedding(texts)
vectorstore.add_elements(chunks,embeddings)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]


In [50]:
class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
        
    def retrieve(self, query: str,top_k:int=5, score_threshold:float=0.0)->List[Dict[str,Any]]:
        
        query_emb=self.embedding_manager.generate_embedding([query])[0]
        try:
            result=self.vector_store.collection.query(
                query_embeddings=[query_emb.tolist()],
                n_results=top_k
                
            )
            print("="*50)
            print(result)
            print("="*50)
            retrieved_docs=[]
            if result['documents'] and result['documents'][0]:
                documents=result['documents'][0]
                metadatas=result['metadatas'][0]
                distances=result['distances'][0]
                ids=result['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate(
                    zip(ids,documents,metadatas,distances)
                ):
                    print(f"Distance: {distance}")

                    similarity_score = distance

                    print(f"Similarity: {similarity_score}")

                    retrieved_docs.append({
                        'id': doc_id,
                        'content': document,
                        'metadata': metadata,
                        'distance': distance,
                        'rank': i + 1
                    })
                print(len(retrieved_docs))
            else:
                print("no document found")
                
            return retrieved_docs          
        except Exception as e:
            print(e)
            return []
        
rag_retriever=RAGRetriever(vectorstore,emb_manager)
rag_retriever

In [52]:
rag_retriever.retrieve("Projects")

Batches: 100%|██████████| 1/1 [00:00<00:00, 35.58it/s]

{'ids': [['doc_3099f337_19', 'doc_37bac9b3_20', 'doc_df4b1a40_21', 'doc_a6dcfb48_13', 'doc_5708766e_13']], 'embeddings': None, 'documents': [['MOHD. MUSHEER \nMachine Learning Engineer | ML Systems & HPC \n+91 9370086707 |  musheerayan@gmail.com | musheer.me |  LinkedIn | GitHub | Amravati, Maharashtra \nPROFESSIONAL SUMMARY \nMachine Learning Engineer with strong foundations in machine learning, deep learning, and model optimization. Experienced in \nbuilding and deploying scalable ML systems using Python and PyTorch, with hands-on work in NLP, transformer models, and \nreal-world applications. Passionate about solving practical problems through efficient, production-ready AI solutions.\nEDUCATION \nBachelor of Computer Applications (BCA) \nTakshashila Mahavidyalaya, Amravati \nExpected Graduation: May 2026\nTECHNICAL SKILLS \nProgramming: Python, SQL, R, C++ \nMachine Learning: Linear Models, Tree-Based Models, Ensemble Methods, Optimization Algorithms, Model Evaluation \nHigh-Perfor

[{'id': 'doc_3099f337_19',
  'content': 'MOHD. MUSHEER \nMachine Learning Engineer | ML Systems & HPC \n+91 9370086707 |  musheerayan@gmail.com | musheer.me |  LinkedIn | GitHub | Amravati, Maharashtra \nPROFESSIONAL SUMMARY \nMachine Learning Engineer with strong foundations in machine learning, deep learning, and model optimization. Experienced in \nbuilding and deploying scalable ML systems using Python and PyTorch, with hands-on work in NLP, transformer models, and \nreal-world applications. Passionate about solving practical problems through efficient, production-ready AI solutions.\nEDUCATION \nBachelor of Computer Applications (BCA) \nTakshashila Mahavidyalaya, Amravati \nExpected Graduation: May 2026\nTECHNICAL SKILLS \nProgramming: Python, SQL, R, C++ \nMachine Learning: Linear Models, Tree-Based Models, Ensemble Methods, Optimization Algorithms, Model Evaluation \nHigh-Performance Computing: BLAS, LAPACK, OpenMP, Memory Optimization, Zero-Copy Data Exchange',
  'metadata': {'

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
from langchain.chat_models import init_chat_model

llm=init_chat_model(model="groq:openai/gpt-oss-120b")

In [55]:
def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

answer=rag_simple("What are Projects of Mohd. Musheer resume?",rag_retriever,llm)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 38.66it/s]


{'ids': [['doc_3099f337_19', 'doc_37bac9b3_20', 'doc_58c9b5e9_22']], 'embeddings': None, 'documents': [['MOHD. MUSHEER \nMachine Learning Engineer | ML Systems & HPC \n+91 9370086707 |  musheerayan@gmail.com | musheer.me |  LinkedIn | GitHub | Amravati, Maharashtra \nPROFESSIONAL SUMMARY \nMachine Learning Engineer with strong foundations in machine learning, deep learning, and model optimization. Experienced in \nbuilding and deploying scalable ML systems using Python and PyTorch, with hands-on work in NLP, transformer models, and \nreal-world applications. Passionate about solving practical problems through efficient, production-ready AI solutions.\nEDUCATION \nBachelor of Computer Applications (BCA) \nTakshashila Mahavidyalaya, Amravati \nExpected Graduation: May 2026\nTECHNICAL SKILLS \nProgramming: Python, SQL, R, C++ \nMachine Learning: Linear Models, Tree-Based Models, Ensemble Methods, Optimization Algorithms, Model Evaluation \nHigh-Performance Computing: BLAS, LAPACK, OpenMP,

In [58]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['distance'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    best_distance = min(doc['distance'] for doc in results)    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'best_distance': best_distance
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What are Projects of Mohd. Musheer resume?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Context Preview:", result['context'][:300])

Batches: 100%|██████████| 1/1 [00:00<00:00, 45.89it/s]


{'ids': [['doc_3099f337_19', 'doc_37bac9b3_20', 'doc_58c9b5e9_22']], 'embeddings': None, 'documents': [['MOHD. MUSHEER \nMachine Learning Engineer | ML Systems & HPC \n+91 9370086707 |  musheerayan@gmail.com | musheer.me |  LinkedIn | GitHub | Amravati, Maharashtra \nPROFESSIONAL SUMMARY \nMachine Learning Engineer with strong foundations in machine learning, deep learning, and model optimization. Experienced in \nbuilding and deploying scalable ML systems using Python and PyTorch, with hands-on work in NLP, transformer models, and \nreal-world applications. Passionate about solving practical problems through efficient, production-ready AI solutions.\nEDUCATION \nBachelor of Computer Applications (BCA) \nTakshashila Mahavidyalaya, Amravati \nExpected Graduation: May 2026\nTECHNICAL SKILLS \nProgramming: Python, SQL, R, C++ \nMachine Learning: Linear Models, Tree-Based Models, Ensemble Methods, Optimization Algorithms, Model Evaluation \nHigh-Performance Computing: BLAS, LAPACK, OpenMP,

In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['distance'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Batches: 100%|██████████| 1/1 [00:00<00:00, 70.07it/s]

{'ids': [['doc_2b01c043_10', 'doc_f3798272_10', 'doc_64ef2d36_10']], 'embeddings': None, 'documents': [["massive reduction for zero code changes — just making sure your prompt structure is\ncache-friendly.\nTo maximize cache hits: - Batch related requests within 5-minute windows so they all hit\nthe cache - Don't update system prompt files mid-session — changes invalidate the cache\n- Keep reference docs in the prompt prefix (stable) and user queries at the end (variable)\nTip 5: Summarize Instead of Stuffing\nOne of the most common token sinks is cramming too much raw content into context. Full\nconversation histories, entire retrieved documents, verbose outputs from previous pipeline\nsteps — all of it costs tokens, and most of it is noise.\nConversation history:\nDon't pass your entire chat transcript with every message. Keep the last 3-5 messages\nverbatim (the model needs recent context) and summarize everything before that. A 200-\ntoken summary of what was discussed earlier repl

s so they all hit
the cache - Don't update system prompt files mid-session — changes invalidate the cache
- Keep reference docs in the prompt prefix (stable) and user queries at the end (variable)
Tip 5: Summarize Instead of Stuffing
One of the most common token sinks is cramming too much raw content into context. Full
conversation histories, entire retrieved documents, verbose outputs from previous pipeline
steps — all of it costs tokens, and most of it is noise.
Conversation history:
Don't pass your entire chat transcript with every message. Keep the last 3-5 messages
verbatim (the model needs recent context) and summarize everything before that. A 200-
token summary of what was discussed earlier replaces 5,000+ tokens of raw message
history.

massive reduction for zero code changes — just making sure your prompt structure is
cache-friendly.
To maximize cache hits: - Batch related requests within 5-minute windows so they all hit
the cache - Don't update system prompt files mid-sessio